In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa import stattools
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima.arima import auto_arima

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (15, 6)
sns.set_style("whitegrid")


In [2]:
def evaluate_forecast(y_true, y_pred, model_name="Model"):
    compare_df = pd.concat(
        [pd.Series(y_true, name="y_true"), pd.Series(y_pred, name="y_pred")],
        axis=1
    ).dropna()

    mae = mean_absolute_error(compare_df["y_true"], compare_df["y_pred"])
    rmse = np.sqrt(mean_squared_error(compare_df["y_true"], compare_df["y_pred"]))
    mape = np.mean(
        np.abs((compare_df["y_true"] - compare_df["y_pred"]) / compare_df["y_true"])
    ) * 100

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE(%)": mape
    }


## 1. 讀取資料

In [ ]:
DATA_PATH = "2330_tw_daily.csv"

if os.path.exists(DATA_PATH):
    Price = pd.read_csv(DATA_PATH)
    print(f"讀取本地檔案：{DATA_PATH}")
else:
    import yfinance as yf
    Price = yf.download("2330.TW", start="2015-01-01", auto_adjust=False)
    Price = Price.reset_index()
    Price.to_csv(DATA_PATH, index=False)
    print(f"已下載資料並儲存為：{DATA_PATH}")

Price.head()


[*********************100%***********************]  1 of 1 completed

已下載資料並儲存為：2330_tw_daily.csv


Price,Date,Adj Close,Close,High,Low,Open,Volume
Ticker,,2330.TW,2330.TW,2330.TW,2330.TW,2330.TW,2330.TW
0,2015-01-05,102.630020,139.5,140.5,137.5,140.5,32046000
1,2015-01-06,98.215805,133.5,137.5,133.0,137.5,66778000
2,2015-01-07,98.583672,134.0,135.0,133.5,133.5,43703000
3,2015-01-08,101.526466,138.0,138.0,136.0,136.5,42491000
4,2015-01-09,98.583672,134.0,135.5,133.0,135.0,61558000


In [4]:
# 檢查欄位與缺失值
print(Price.columns.tolist())
print("\n缺失值數量：")
print(Price.isna().sum())


[('Date', ''), ('Adj Close', '2330.TW'), ('Close', '2330.TW'), ('High', '2330.TW'), ('Low', '2330.TW'), ('Open', '2330.TW'), ('Volume', '2330.TW')]

缺失值數量：
Price      Ticker 
Date                  0
Adj Close  2330.TW    0
Close      2330.TW    0
High       2330.TW    0
Low        2330.TW    0
Open       2330.TW    0
Volume     2330.TW    0
dtype: int64


In [5]:
# 僅保留日期與收盤價，整理成時間序列格式
# 有些 yfinance 版本欄名可能是 Close 或 ('Close', '2330.TW')，這裡做較穩健的處理

close_col = None
for col in Price.columns:
    if str(col).lower() == "close":
        close_col = col
        break

if close_col is None:
    # MultiIndex 欄位時，尋找最上層為 Close 的欄位
    for col in Price.columns:
        if isinstance(col, tuple) and str(col[0]).lower() == "close":
            close_col = col
            break

if close_col is None:
    raise KeyError("找不到 Close 欄位，請檢查下載資料格式。")

date_col = "Date" if "Date" in Price.columns else Price.columns[0]

Price_2330 = Price[[date_col, close_col]].copy()
Price_2330.columns = ["Date", "2330 臺積電"]
Price_2330["Date"] = pd.to_datetime(Price_2330["Date"])
Price_2330 = Price_2330.sort_values("Date").set_index("Date")

Price_2330.head()


KeyError: "[('Close', '2330.TW')] not in index"

In [ ]:
# 新增日報酬率
Price_2330["Return"] = Price_2330["2330 臺積電"].pct_change()
Price_2330 = Price_2330.dropna().copy()

Price_2330.head()


## 2. 時間序列分析

In [ ]:
# 收盤價走勢
Price_2330["2330 臺積電"].plot(title="TSMC (2330) Daily Close Price")
plt.ylabel("Price")
plt.show()

# 日報酬率走勢
Price_2330["Return"].plot(title="TSMC (2330) Daily Return")
plt.axhline(0, linestyle="--")
plt.ylabel("Return")
plt.show()


## 3. 自相關性（ACF / PACF）

In [ ]:
# 以日報酬率觀察自相關性
acfs = stattools.acf(Price_2330["Return"], nlags=20)
pacfs = stattools.pacf(Price_2330["Return"], nlags=20)

print("ACF:", acfs)
print("\nPACF:", pacfs)


In [ ]:
plot_acf(Price_2330["Return"], lags=20)
plt.title("ACF of Daily Return")
plt.show()

plot_pacf(Price_2330["Return"], lags=20)
plt.title("PACF of Daily Return")
plt.show()


## 4. 切分訓練集與驗證集

In [ ]:
# 以最後 20% 資料作為驗證集
split_idx = int(len(Price_2330) * 0.8)

train = Price_2330.iloc[:split_idx].copy()
valid = Price_2330.iloc[split_idx:].copy()

training = train["2330 臺積電"]
validation = valid["2330 臺積電"]

print("Train size:", len(train))
print("Valid size:", len(valid))
print("Train period:", train.index.min().date(), "to", train.index.max().date())
print("Valid period:", valid.index.min().date(), "to", valid.index.max().date())


## 5. Auto ARIMA

In [ ]:
# 對每日股價先使用非季節型 ARIMA
# 若之後想研究週期性，可再嘗試 seasonal=True 與不同 m 值

arima_model = auto_arima(
    training,
    start_p=0, start_q=0,
    max_p=3, max_q=3,
    d=None,
    test="adf",
    seasonal=False,
    trace=True,
    error_action="ignore",
    suppress_warnings=True,
    stepwise=True
)

arima_model.summary()


In [ ]:
arima_forecast = pd.Series(
    arima_model.predict(n_periods=len(valid)),
    index=valid.index,
    name="Prediction"
)

arima_forecast.head()


In [ ]:
arima_result = evaluate_forecast(
    valid["2330 臺積電"],
    arima_forecast,
    "Auto ARIMA"
)

arima_result


In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(arima_forecast, label="ARIMA Forecast")
plt.title("Auto ARIMA Forecast - TSMC Daily Close Price")
plt.legend()
plt.show()


## 6. ADF test

In [ ]:
# 用報酬率做平穩性檢定通常比直接用價格更合理
adf_result = adfuller(Price_2330["Return"].dropna())

print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Critical Values:")
for key, value in adf_result[4].items():
    print(f"  {key}: {value}")


## 7. Residual Analysis（ARIMA）

In [ ]:
arima_residuals = valid["2330 臺積電"] - arima_forecast

plt.figure(figsize=(15, 4))
plt.plot(arima_residuals)
plt.title("ARIMA Residuals")
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(arima_residuals.dropna(), bins=30)
plt.title("Residual Distribution")
plt.show()

plot_acf(arima_residuals.dropna(), lags=30)
plt.title("Residual ACF")
plt.show()


## 8. Naive Forecast

In [ ]:
# Naive forecast: 下一天 = 前一天實際值
naive_pred = valid["2330 臺積電"].shift(1)
naive_pred.iloc[0] = train["2330 臺積電"].iloc[-1]

naive_result = evaluate_forecast(
    valid["2330 臺積電"],
    naive_pred,
    "Naive Forecast"
)

plt.figure(figsize=(16, 6))
plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(naive_pred, label="Naive Forecast")
plt.title("Naive Forecast")
plt.legend()
plt.show()

naive_result


## 9. Moving Average Forecast

In [ ]:
# 使用最近 20 個交易日平均作為預測
window = 20
history = list(train["2330 臺積電"])
ma_pred = []

for actual in valid["2330 臺積電"]:
    pred = np.mean(history[-window:])
    ma_pred.append(pred)
    history.append(actual)

ma_pred = pd.Series(ma_pred, index=valid.index, name="MA Prediction")

ma_result = evaluate_forecast(
    valid["2330 臺積電"],
    ma_pred,
    f"Moving Average ({window} days)"
)

plt.figure(figsize=(16, 6))
plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(ma_pred, label="Moving Average Forecast")
plt.title("Moving Average Forecast")
plt.legend()
plt.show()

ma_result


## 10. Exponential Smoothing

In [ ]:
es_model = ExponentialSmoothing(
    train["2330 臺積電"],
    trend=None,
    seasonal=None
)

es_fit = es_model.fit()
es_pred = pd.Series(
    es_fit.forecast(len(valid)),
    index=valid.index,
    name="ES Prediction"
)

es_result = evaluate_forecast(
    valid["2330 臺積電"],
    es_pred,
    "Exponential Smoothing"
)

plt.figure(figsize=(16, 6))
plt.plot(train["2330 臺積電"], label="Train")
plt.plot(valid["2330 臺積電"], label="Valid")
plt.plot(es_pred, label="Exponential Smoothing Forecast")
plt.title("Exponential Smoothing Forecast")
plt.legend()
plt.show()

es_result


## 11. 模型比較

In [ ]:
result_df = pd.DataFrame([
    naive_result,
    ma_result,
    es_result,
    arima_result
])

result_df.sort_values("RMSE").reset_index(drop=True)
